In [ ]:
# Get significant features from ANOVA and Tukey results
significant_features = []
significant_pairs = {}

# Read ANOVA results
with open(os.path.join(root_dir, 'Output/SPR-2025/anova_results.txt'), 'r') as f:
    anova_text = f.read()
    
# Read Tukey results    
with open(os.path.join(root_dir, 'Output/SPR-2025/tukey_results.txt'), 'r') as f:
    tukey_text = f.read()

# Only include features that have significant post-hoc differences
for section in tukey_text.split('Feature: ')[1:]:
    feature_name = section.split('\n')[0]
    if 'reject' in section:
        pairs = []
        for line in section.split('\n'):
            if 'True' in line:
                parts = line.split()
                pairs.append((int(parts[0]), int(parts[1])))
        if pairs:  # Only add if there are significant pairs
            significant_pairs[feature_name] = pairs
            significant_features.append(feature_name)

# Group features by type
feature_groups = {
    'Offset': [f for f in significant_features if 'offset' in f.lower()],
    'Exponent': [f for f in significant_features if 'exponent' in f.lower()],
    'Delta': [f for f in significant_features if 'delta' in f.lower()],
    'Theta': [f for f in significant_features if 'theta' in f.lower()],
    'Alpha': [f for f in significant_features if 'alpha' in f.lower()],
    'Beta': [f for f in significant_features if 'beta' in f.lower()],
    'Low Gamma': [f for f in significant_features if 'lowgamma' in f.lower()],
    'High Gamma': [f for f in significant_features if 'highgamma' in f.lower()],
    'Relative': [f for f in significant_features if 'relative' in f.lower()],
    'Periodic': [f for f in significant_features if 'periodic' in f.lower()]
}

# Add "Other" category for features that don't fit existing categories
categorized_features = [f for group in feature_groups.values() for f in group]
other_features = [f for f in significant_features if f not in categorized_features]
if other_features:
    feature_groups['Other'] = other_features

# Create separate figures for each feature group
for group_name, features in feature_groups.items():
    if not features:  # Skip empty groups
        continue
        
    n_features = len(features)
    n_cols = 2
    n_rows = (n_features + 1) // 2  # Round up division

    fig = plt.figure(figsize=(12, n_rows * 4))
    fig.suptitle(f'{group_name} Features', fontsize=16, y=1.02)

    for idx, feature in enumerate(features):
        plt.subplot(n_rows, n_cols, idx + 1)
        
        # Create boxplot
        sns.boxplot(data=df, x='diagnostic_group', y=feature)
        
        # Add individual points
        sns.stripplot(data=df, x='diagnostic_group', y=feature, 
                     color='red', alpha=0.3, size=4)
        
        # Add significance bars
        y_max = df[feature].max()
        y_min = df[feature].min()
        y_range = y_max - y_min
        bar_height = y_range * 0.05
        
        for i, (group1, group2) in enumerate(significant_pairs[feature]):
            y_pos = y_max + (i + 1) * bar_height * 1.5
            
            plt.plot([group1, group2], [y_pos, y_pos], 'k-')
            plt.plot([group1, group1], [y_pos-bar_height/2, y_pos], 'k-')
            plt.plot([group2, group2], [y_pos-bar_height/2, y_pos], 'k-')
            plt.text((group1 + group2)/2, y_pos + bar_height/2, '*', 
                    horizontalalignment='center')
        
        plt.ylim(y_min - y_range*0.1, y_max + len(significant_pairs[feature])*bar_height*2)
        
        plt.title(feature)
        plt.xlabel('Group')
        plt.xticks([0, 1, 2], ['Control', 'Neurodev', 'Genetic'], rotation=45)

    plt.tight_layout()
    
    # Save each group to a separate PDF
    plt.savefig(os.path.join(root_dir, f'Output/SPR-2025/significant_features_{group_name.lower().replace(" ","_")}_boxplots.pdf'))
    plt.close()

# Create a summary of feature groups
print("\nFeature Groups Summary:")
print("-" * 50)
for group_name, features in feature_groups.items():
    if features:
        print(f"\n{group_name} Features ({len(features)}):")
        for feature in features:
            print(f"  - {feature}")


In [ ]:
# Separate analysis based on assumptions
def analyze_feature(feature, data):
    if feature in features_all_ok:
        # Parametric test (ANOVA)
        model = ols(f'{feature} ~ diagnostic_group', data=data).fit()
        return anova_lm(model, typ=2)
    else:
        # Non-parametric test (Kruskal-Wallis)
        groups = [group for _, group in data.groupby('diagnostic_group')[feature]]
        stat, p = stats.kruskal(*groups)
        return pd.DataFrame({'statistic': [stat], 'p-value': [p]})

# Run analysis for all features
results = {}
for feature in features_all_ok + features_normality_violated + features_homogeneity_violated:
    results[feature] = analyze_feature(feature, diagnostic_groups_data)

# Save results
with open(os.path.join(root_dir, 'Output/SPR-2025/analysis_results.txt'), 'w') as f:
    for feature, result in results.items():
        f.write(f"\n{feature}:\n{result}\n")

In [ ]:
anova_df = pd.DataFrame(anova_results).T  # Transpose to make features as rows
anova_df.sort_values('p-value', inplace=True)

# --- Filter significant features ---
signif_features = anova_df[anova_df['p-value'] < 0.05].index.tolist()

# --- Run Tukey HSD for each significant feature ---
for feature in signif_features:
    print(f"\n📊 Tukey HSD for: {feature}")
    tukey = pairwise_tukeyhsd(endog=df[feature], groups=df['diagnostic_group'], alpha=0.05)
    print(tukey.summary())

# Save Tukey HSD results to text file
with open(os.path.join(root_dir, 'Output/SPR-2025/tukey_results.txt'), 'w') as f:
    f.write("Tukey HSD Results Summary\n")
    f.write("=======================\n\n")
    
    for feature in signif_features:
        f.write(f"\nFeature: {feature}\n")
        f.write("-------------------\n")
        tukey = pairwise_tukeyhsd(endog=df[feature], groups=df['diagnostic_group'], alpha=0.05)
        f.write(tukey.summary().as_text())
        f.write("\n" + "="*50 + "\n")

print("\nTukey HSD results exported to 'tukey_results.txt'")
